# Fine-tuning — Classification par domaine juridique (Étape 5)

Ce notebook fine-tune un transformer multilingue pour prédire le **domaine juridique**
d'un article à partir du jeu de données relu et corrigé manuellement
(`domain_dataset_final.csv`) — **français ET arabe combinés** cette fois.

**À savoir avant de lancer ce notebook :**

- Le jeu de données contient maintenant **679 exemples fiables** : 134 FR + 545 AR
  (le bug d'extraction PDF arabe — colonnes mélangées, ordre des mots dans les
  lignes RTL, dates mal formées — a été corrigé côté ingestion, puis les 545
  articles arabes ont été relus et corrigés à la main comme la partie FR).
- **12 domaines** au total, dont 3 nouveaux apparus en lisant le corpus arabe :
  **Santé** (formation médicale, recherche biomédicale), **Justice** (huissiers,
  notaires/adouls, juges de liaison), **Enseignement** (gouvernance des
  établissements, équivalences de diplômes). Le domaine **Télécommunications**
  identifié précédemment côté français n'apparaît pas dans le jeu de données
  actuel (le document source qui le contenait n'est plus dans le corpus traité
  cette fois) — il réapparaîtra automatiquement si ce document est retraité.
- Le domaine **"Pénal"** (3 exemples) et **"Social"** (2 exemples) restent
  exclus automatiquement (seuil `MIN_SAMPLES_PER_CLASS`) — toujours trop peu de
  données pour les évaluer ou les entraîner sérieusement.
- **679 exemples reste petit** pour un fine-tuning de transformer, mais nettement
  mieux qu'avant (283 FR seulement). Ce notebook utilise toujours une
  **validation croisée stratifiée (5 folds)** plutôt qu'un simple split
  train/test, pour une estimation plus fiable de la performance.
- On utilise un modèle **multilingue** (`xlm-roberta-base`), qui gère nativement
  le français et l'arabe dans le même modèle — pas de changement d'architecture
  entre la version FR seule et celle-ci.
- **Recommandation** : les classes "Urbain" (35, presque uniquement FR) et
  "Social" (2, quasi absent) restent sous-représentées — grossir le corpus sur
  ces domaines spécifiquement (traiter plus de PDF les concernant) reste la
  meilleure prochaine étape après ce fine-tuning.


## 1. Installation des dépendances

À exécuter une seule fois par session Colab. **Active un GPU** avant de lancer :
`Exécution > Modifier le type d'exécution > GPU (T4)`.

**Après cette cellule, redémarre la session** (`Exécution > Redémarrer la
session`) avant de continuer -- `transformers` doit être réimporté à froid
pour que le correctif torch/torchvision prenne effet (sinon tu tombes sur
`ModuleNotFoundError: Could not import module 'Trainer'`, causé par un
conflit de version torch/torchvision, pas par une dépendance manquante).


In [ ]:
# IMPORTANT : ne PAS mettre --upgrade sur torch ici -- Colab fournit déjà un
# torch + torchvision compatibles avec son GPU. Les upgrader séparément casse
# le lien entre les deux (erreur 'operator torchvision::nms does not exist'
# au moment d'importer transformers). On upgrade seulement les paquets dont
# on a vraiment besoin, et on retire torchvision (inutile pour du texte) pour
# éviter tout risque de ce genre.
!pip install -q --upgrade transformers datasets accelerate scikit-learn pandas
!pip uninstall -y -q torchvision

# Redémarre le kernel une fois que cette cellule a fini de tourner
# (Exécution > Redémarrer la session), PUIS relance le notebook à partir de
# la cellule suivante -- transformers doit être réimporté à froid pour que
# le correctif prenne effet proprement.


## 2. Chargement du jeu de données

Deux options : upload manuel du CSV, ou lecture depuis Google Drive. Choisis l'une des
deux cellules ci-dessous (laisse l'autre en commentaire).


In [ ]:
# --- Option A : upload manuel (glisse-dépose domain_dataset_corrected.csv) ---
from google.colab import files
uploaded = files.upload()  # sélectionne domain_dataset_corrected.csv
csv_path = list(uploaded.keys())[0]


In [ ]:
# --- Option B : depuis Google Drive (décommente si tu préfères) ---
# from google.colab import drive
# drive.mount('/content/drive')
# csv_path = '/content/drive/MyDrive/projet-nlp-juridique-maroc/domain_dataset_corrected.csv'


In [ ]:
import pandas as pd

df = pd.read_csv(csv_path, encoding='utf-8-sig')
print(f"{len(df)} lignes chargées")
df['label'].value_counts()


## 3. Filtrage et préparation

- On garde le français ET l'arabe (`lang in ('fr', 'ar')`) — les deux ont été
  relus et corrigés manuellement cette fois.
- On retire `"Indéterminé"` (ce n'est pas un vrai domaine, juste "aucun signal trouvé").
- On retire les classes avec moins de `MIN_SAMPLES_PER_CLASS` exemples : impossible à
  évaluer ou entraîner sérieusement dessus (typiquement "Pénal" et "Social").


In [ ]:
MIN_SAMPLES_PER_CLASS = 10

data = df[df['label'] != 'Indéterminé'].copy()

counts = data['label'].value_counts()
kept_labels = counts[counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
dropped_labels = counts[counts < MIN_SAMPLES_PER_CLASS].index.tolist()

if dropped_labels:
    print(f"Domaines exclus (trop peu d'exemples) : {dropped_labels}")
    print("-> à revoir une fois que davantage de documents auront été traités par le pipeline.")

data = data[data['label'].isin(kept_labels)].reset_index(drop=True)
print(f"\n{len(data)} exemples utilisables, {len(kept_labels)} domaines : {kept_labels}")
print(f"  dont FR : {(data['lang']=='fr').sum()}  |  AR : {(data['lang']=='ar').sum()}")
data['label'].value_counts()


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
data['label_id'] = label_encoder.fit_transform(data['label'])
id2label = {i: l for i, l in enumerate(label_encoder.classes_)}
label2id = {l: i for i, l in id2label.items()}
num_labels = len(id2label)

print(id2label)


## 4. Tokenizer et modèle de base

`xlm-roberta-base` : modèle multilingue (100 langues, dont le français et l'arabe),
choisi pour pouvoir réentraîner ce même notebook sur FR+AR une fois le corpus arabe
disponible, sans changer d'architecture.


In [ ]:
import torch
from transformers import AutoTokenizer

MODEL_NAME = "xlm-roberta-base"
MAX_LENGTH = 256  # les articles sont tronqués à 6000 caractères dans le CSV,
                   # mais le contenu utile pour classer le domaine tient largement
                   # dans les premières phrases -> 256 tokens est un compromis
                   # raisonnable vitesse/qualité pour un aussi petit dataset.

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )


## 5. Dataset PyTorch + Trainer avec pondération des classes

Le jeu est déséquilibré (Télécommunications : 101 exemples vs Commercial : 19).
On pondère la fonction de perte (`CrossEntropyLoss`) par l'inverse de la fréquence
de chaque classe, pour éviter que le modèle apprenne juste à toujours prédire la
classe majoritaire.


In [ ]:
import numpy as np
from torch.utils.data import Dataset

class DomainDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize(texts)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


In [ ]:
from transformers import Trainer
import torch.nn as nn

class WeightedTrainer(Trainer):
    """Trainer standard, mais avec une CrossEntropyLoss pondérée par classe
    (compense le déséquilibre Télécommunications=101 vs Commercial=19, etc.)"""
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weight = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss_fct = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }


## 6. Validation croisée stratifiée (5 folds)

Avec ~280 exemples, un seul split train/test donnerait un jeu de test d'une
trentaine de lignes — trop petit pour une estimation fiable. La validation
croisée entraîne 5 modèles séparés (un par fold) et moyenne leurs scores : plus
lent, mais bien plus fiable pour juger si l'approche fonctionne avant de
déployer quoi que ce soit.


In [ ]:
from sklearn.model_selection import StratifiedKFold
from transformers import AutoModelForSequenceClassification, TrainingArguments
import gc

# Garde-fou : si tu vois 'KeyError: label_id' ici, c'est que la cellule de la
# Section 3 (LabelEncoder) n'a pas été exécutée avant celle-ci -- fréquent après
# un redémarrage de session (Section 1) si tu reprends au milieu du notebook au
# lieu de tout rejouer depuis le début. Utilise 'Exécution > Tout exécuter'
# plutôt que de relancer des cellules une par une dans le désordre.
assert "label_id" in data.columns, (
    "Colonne 'label_id' manquante -- relance les cellules de la Section 3 "
    "(LabelEncoder) avant celle-ci, ou utilise Exécution > Tout exécuter."
)

N_FOLDS = 5
EPOCHS = 8
BATCH_SIZE = 8
LEARNING_RATE = 2e-5


In [ ]:
accs = [m["eval_accuracy"] for m in fold_results]
f1s = [m["eval_f1_macro"] for m in fold_results]

print(f"Accuracy moyenne  : {np.mean(accs):.3f} (+/- {np.std(accs):.3f})")
print(f"F1 macro moyen    : {np.mean(f1s):.3f} (+/- {np.std(f1s):.3f})")
print()
print("Rappel : avec ~280 exemples sur 6 classes, un écart-type élevé entre les")
print("folds est normal -- ça reflète surtout la petite taille du dataset, pas")
print("forcément un problème de modèle. Si le F1 macro moyen est proche de celui")
print("du classifieur par mots-clés, le transformer n'apporte pas encore assez")
print("de valeur pour justifier sa complexité -- grossir le corpus (plus de PDF")
print("traités) avant de retenter est la meilleure prochaine étape dans ce cas.")


## 7. Modèle final (déploiement)

Une fois la validation croisée jugée satisfaisante, on entraîne un dernier modèle
sur **100% des données utilisables** (train + val réunis) — c'est celui qu'on
sauvegarde et qu'on utilise en production. Sa performance sur ses propres données
d'entraînement n'est PAS une mesure fiable de qualité : se référer aux scores de
la validation croisée ci-dessus pour juger si le modèle est prêt.


In [ ]:
full_ds = DomainDataset(X, y)

class_counts = np.bincount(y, minlength=num_labels)
class_weights = torch.tensor(
    [len(y) / (num_labels * max(c, 1)) for c in class_counts], dtype=torch.float
)

final_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels, id2label=id2label, label2id=label2id
)

final_args = TrainingArguments(
    output_dir="/content/final_model",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    save_strategy="no",
    logging_strategy="epoch",
    report_to="none",
    seed=42,
)

final_trainer = WeightedTrainer(
    class_weights=class_weights,
    model=final_model,
    args=final_args,
    train_dataset=full_ds,
    compute_metrics=compute_metrics,
)

final_trainer.train()


In [ ]:
SAVE_DIR = "/content/domain_classifier_fr_ar"

final_trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

import json as _json
with open(f"{SAVE_DIR}/id2label.json", "w", encoding="utf-8") as f:
    _json.dump(id2label, f, ensure_ascii=False, indent=2)

print(f"Modèle sauvegardé dans {SAVE_DIR}")


In [ ]:
# Compresse le modèle pour le télécharger ou le copier vers Google Drive
!zip -rq /content/domain_classifier_fr_ar.zip /content/domain_classifier_fr_ar

from google.colab import files
files.download("/content/domain_classifier_fr_ar.zip")

# Alternative : copier vers Drive au lieu de télécharger
# !cp /content/domain_classifier_fr_ar.zip /content/drive/MyDrive/projet-nlp-juridique-maroc/


## 8. Inférence — comparaison avec le classifieur par mots-clés

Fonction prête à l'emploi pour classer un nouveau texte, et comparaison rapide
avec `keyword_classifier.classify_text()` sur quelques exemples — pour juger
concrètement si le transformer apporte quelque chose par rapport à l'existant
(cf. décision prise : garder le classifieur mots-clés en parallèle / fallback).


In [ ]:
def classify_text_transformer(text: str) -> dict:
    """Retourne {"label": ..., "scores": {domaine: probabilité}}."""
    device = next(final_model.parameters()).device  # le modèle est sur GPU
                                                       # après l'entraînement --
                                                       # il faut y envoyer aussi
                                                       # les tenseurs d'entrée,
                                                       # sinon erreur de device
                                                       # mismatch (cpu vs cuda).
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = final_model(**inputs).logits
    probs = torch.softmax(logits, dim=1)[0]
    pred_id = int(torch.argmax(probs))
    return {
        "label": id2label[pred_id],
        "scores": {id2label[i]: float(probs[i]) for i in range(num_labels)},
    }


In [ ]:
# Exemples rapides (FR + AR) -- remplace par tes propres textes pour tester
samples = [
    "Est fixé dans l'annexe joint au présent arrêté conjoint le cahier des "
    "charges relatif aux spécifications techniques minimales des "
    "infrastructures de télécommunications.",
    "Le montant de l'amende de transaction est fixé par l'administration des "
    "eaux et forêts en fonction de la gravité de l'infraction constatée.",
    "يخضع المفوض القضائي كل سنة لدورة على الاقل من دورات التكوين المستمر.",
    "تحدث لجنة علمية بالمعهد تتولى ابداء الراي في البرامج البيداغوجية "
    "وموضوعات البحث العلمي.",
]

for s in samples:
    result = classify_text_transformer(s)
    top3 = sorted(result["scores"].items(), key=lambda kv: kv[1], reverse=True)[:3]
    print(f"Texte : {s[:80]}...")
    print(f"  -> Transformer : {result['label']}  (top 3 : {top3})")
    print()


## Prochaines étapes

1. **Grossir le corpus** sur les domaines encore sous-représentés — "Urbain"
   (35, quasi uniquement FR) et "Social" (2, exclu de cet entraînement) en
   particulier — en traitant davantage de PDF du Bulletin Officiel via
   `run_extraction` / `run_consolidation` / `build_training_dataset`.
2. **Réintégrer "Pénal"** une fois qu'il y a plus d'exemples (au moins une
   dizaine) — toujours exclu faute de données suffisantes.
3. **Retraiter le document source du domaine "Télécommunications"** (identifié
   précédemment côté français) si tu veux qu'il réapparaisse dans un futur
   entraînement — il n'est pas dans le corpus actuel.
4. **Comparer sérieusement** avec `keyword_classifier.py` sur ce même jeu de
   test (FR et AR désormais) avant de remplacer quoi que ce soit en
   production : si le transformer n'apporte pas un gain net de F1 macro sur
   les deux langues, garder le classifieur mots-clés (plus simple, plus
   rapide, pas de dépendance GPU) reste la décision la plus raisonnable.
